# 🎙️ Enstruct Google Colab Demo
### **Transcribe. Structure. Free.**

Welcome to the official **Enstruct** Google Colab Demo! Enstruct is a professional open-source toolkit wrapping OpenAI's Whisper model (via `faster-whisper`) to provide fast, high-quality, and completely free audio transcription and translation.

In this notebook, we will walk you through:
1. **Environment Setup**: Installing `enstruct` and `faster-whisper`.
2. **Google Drive Integration**: Optional mounting of your Google Drive to transcribe files directly from your cloud storage.
3. **Single File Transcription**: Transcribing a sample audio file, detecting its language, and examining the output segments.
4. **Translation to English**: Translating non-English audio files directly into English transcripts.
5. **Formatting Subtitles**: Generating SRT, VTT, and TXT outputs from transcription results.

## 🛠️ Step 1: Install Dependencies

First, we will install the `enstruct` package along with the `faster-whisper` backend. We also ensure ffmpeg is installed, which is required for processing various audio and video container formats.

In [ ]:
# Install the enstruct package and faster-whisper
!pip install enstruct faster-whisper

# Ensure ffmpeg is installed (usually pre-installed on Google Colab instances)
!apt-get update -qq && apt-get install -y -qq ffmpeg

## 📁 Step 2: (Optional) Mount Google Drive

If you have audio files stored on Google Drive, you can mount your drive to access them seamlessly. Un-comment and run the cell below to connect your drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 🎵 Step 3: Create Sample Audio & Run Transcription

Let's create a quick mock/sample audio file or download a public audio clip for testing. For demonstration purposes, we will synthesize a simple silent audio file using ffmpeg to ensure the pipeline runs without requiring actual microphone input.

In [ ]:
# Generate a 5-second sample audio file using ffmpeg containing a 440Hz sine wave tone
!ffmpeg -f lavfi -i "sine=frequency=440:duration=5" -c:a pcm_s16le sample.wav -y

import os
print("Sample audio created:", os.path.exists("sample.wav"))

### Using the Enstruct API to Transcribe

We will now import `EnstructTranscriber` from `enstruct.core.transcriber` and initialize it. 
By default, `device` is set to `auto` which auto-detects if GPU is available (highly recommended in Google Colab, change runtime type to T4 GPU for best performance).

In [ ]:
from enstruct.core.transcriber import EnstructTranscriber

# Initialize transcriber. Use tiny model for fast loading inside this notebook demonstration.
transcriber = EnstructTranscriber(model_size="tiny", device="auto")

# Transcribe the sample wav file
# Note: Since it is just a tone, the transcription may be empty or contain minor noise output.
result = transcriber.transcribe("sample.wav")

print("--- Transcription Summary ---")
print("Detected Language:", result["language"])
print("Audio Duration (seconds):", result["duration"])
print("Transcribed Text:", result["text"])
print("Segments Count:", len(result["segments"]))

### Language Detection Only

If you only want to know the spoken language without fully transcribing the audio, you can use the lightweight `detect_language` method.

In [ ]:
language_code = transcriber.detect_language("sample.wav")
print("Detected Language Code:", language_code)

## 🌐 Step 4: Translation to English

Enstruct offers an `EnstructTranslator` class which directly translates non-English audio files into English transcripts. This wraps the Whisper `translate` task.

In [ ]:
from enstruct.core.translator import EnstructTranslator

# Initialize the translator
translator = EnstructTranslator(model_size="tiny", device="auto")

# Translate target audio (e.g., sample.wav)
translation_result = translator.translate("sample.wav")
print("Translated text:", translation_result["text"])

## 📝 Step 5: Formatting and Exporting Subtitles

Enstruct makes it extremely easy to generate SRT and VTT subtitle files, as well as plain TXT files from transcription segments.

In [ ]:
from enstruct.tools.subtitle import SubtitleGenerator

# Sample segments (mimicking Whisper outputs)
sample_segments = [
    {"start": 0.0, "end": 2.5, "text": "Welcome to the open-source Enstruct toolkit."},
    {"start": 2.5, "end": 5.0, "text": "Start transcribing and translating your audio for free!"}
]

generator = SubtitleGenerator()

# Generate SRT subtitles
generator.generate(sample_segments, "output.srt", format="srt")
print("--- Generated SRT Subtitles ---")
with open("output.srt", "r") as f:
    print(f.read())

# Generate VTT subtitles
generator.generate(sample_segments, "output.vtt", format="vtt")
print("--- Generated VTT Subtitles ---")
with open("output.vtt", "r") as f:
    print(f.read())

## 💻 Running via CLI

Enstruct includes a convenient CLI script. You can run all the operations shown above directly in the command-line interface! Try these commands:

In [ ]:
# Let's check the CLI help
!enstruct --help

In [ ]:
# Transcribe via CLI
!enstruct transcribe sample.wav --format srt --output cli_output.srt

print("\n--- CLI Output File exists:", os.path.exists("cli_output.srt"))

--- 
### 🎉 Congratulations! You are now ready to build and transcribe on your own using **Enstruct**!
Feel free to submit issues, pull requests, and star our repository on [GitHub](https://github.com/enstruct/enstruct).